# Gemma 3 1B Corporate Chatbot - QLoRA Fine-tuning (Colab)

This notebook is a thin wrapper around the scripts in this project. All the
actual logic (dataset validation, splitting, training, testing) lives in
`scripts/` and `src/corporate_chatbot/` - this notebook just calls those
scripts in order so you can run the whole pipeline on a Colab GPU runtime.

**Before running:** In Colab, go to `Runtime > Change runtime type` and
select a GPU (e.g. T4). QLoRA training requires a CUDA GPU.

Steps: install deps -> authenticate with Hugging Face -> get the project
onto the runtime -> validate dataset -> (optional) regenerate dataset ->
train -> evaluate -> test.

## 1. Install dependencies

In [ ]:
!pip install -q -r requirements.txt

## 2. Get the project onto this runtime

If you're running this notebook from within a clone of the project already
(e.g. opened directly from your repo), skip this cell. Otherwise, clone the
repository below.

In [ ]:
REPO_URL = "https://github.com/saadalikhan02/finetune-chatbot.git"
!git clone $REPO_URL
%cd finetune-chatbot

## 3. Authenticate with Hugging Face

`google/gemma-3-1b-it` is a gated model. Accept the license at
https://huggingface.co/google/gemma-3-1b-it, then authenticate below. You
can either paste a token into the interactive login prompt (not stored in
the notebook), or set `HF_TOKEN` as a Colab secret and load it from there.

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

## 4. Validate the dataset

This repo already includes a real, grounded dataset generated by crawling
https://technyxsystems.com/ (see `data/website/README.md` for the full
crawl -> clean -> facts -> QA -> split pipeline). `configs/training.yaml`
points at these files already - just validate them before training.

In [ ]:
!python scripts/validate_dataset.py --input data/datasets/train.jsonl
!python scripts/validate_dataset.py --input data/datasets/validation.jsonl
!python scripts/validate_dataset.py --input data/datasets/test.jsonl

## 5. (Optional) Re-run the website pipeline

Only needed if the Technyx site has changed since this dataset was
generated, or you want to regenerate it from scratch. Skip this cell
otherwise - `data/datasets/` already has what `configs/training.yaml`
needs.

In [ ]:
# !python scripts/webdata/crawl_site.py
# !python scripts/webdata/clean_pages.py
# !python scripts/webdata/extract_facts.py
# !python scripts/webdata/generate_qa.py
# !python scripts/webdata/seed_curated_qa.py
# !python scripts/webdata/seed_edge_case_qa.py
# !python scripts/webdata/validate_grounding.py
# !python scripts/webdata/build_datasets.py
# !python scripts/webdata/export_eval_cases.py

## 6. Train

Uses `configs/training.yaml` and `configs/lora.yaml`. Edit those files (or
pass different `--config`/`--lora-config` paths) to change hyperparameters.

In [ ]:
!python scripts/train.py --config configs/training.yaml --lora-config configs/lora.yaml

## 7. Evaluate: base model vs fine-tuned model

In [ ]:
!python scripts/evaluate.py --adapter outputs/gemma3-1b-corporate-lora --test-cases evaluation/test_cases.jsonl

## 8. Interactive test

`scripts/test_model.py` is interactive (reads from stdin in a loop), which
doesn't work well inside a notebook cell. Run it from a Colab terminal
instead (Colab Pro) or download the adapter and run it locally:

```bash
python scripts/test_model.py --adapter outputs/gemma3-1b-corporate-lora
```